# Metric Learning: Training Embedding Spaces

| | |
|---|---|
| **Level** | Tier 3: Advanced Guide |
| **Time** | ~30 minutes |
| **Prerequisites** | `05_composition.py`, embedding space concepts |
| **Metrics covered** | ContrastiveLoss, TripletMarginLoss, NTXentLoss, ArcFaceLoss |
| **Key concepts** | Contrastive learning, hard negative mining, angular margins |

In [ ]:
"""Metric learning losses: differentiable distance functions for embedding spaces.

Demonstrates:
- ContrastiveLoss on embedding pairs
- TripletMarginLoss on anchor/positive/negative
- NTXentLoss (InfoNCE) for self-supervised learning
- ArcFaceLoss with learnable weight matrix
- Verifying gradients flow with jax.grad()
- Mining hard negatives with HardNegativeMiner
"""

import flax.nnx as nnx
import jax
import jax.numpy as jnp

from calibrax.metrics.learning import (
    ArcFaceLoss,
    ContrastiveLoss,
    HardNegativeMiner,
    NTXentLoss,
    TripletMarginLoss,
)


def main() -> None:
    """Run metric learning loss examples."""
    key = jax.random.PRNGKey(42)

    # -- Shared batch: 8 embeddings in R^16, 4 classes (2 per class) -------
    embedding_dim = 16
    num_samples = 8
    key, subkey = jax.random.split(key)
    embeddings = jax.random.normal(subkey, (num_samples, embedding_dim))
    labels = jnp.array([0, 0, 1, 1, 2, 2, 3, 3])

    print("=== Shared Data ===")
    print(f"  Embeddings shape: {embeddings.shape}")
    print(f"  Labels: {labels.tolist()}")

    # -- 1. ContrastiveLoss ------------------------------------------------
    print("\n=== ContrastiveLoss ===")
    contrastive = ContrastiveLoss(margin=1.0)
    loss_val = contrastive(embeddings, labels)
    print(f"  Loss (margin=1.0): {float(loss_val):.6f}")

    # Compare different margins
    for margin in [0.5, 1.0, 2.0]:
        cl = ContrastiveLoss(margin=margin)
        val = cl(embeddings, labels)
        print(f"  margin={margin:.1f}  loss={float(val):.6f}")

    # -- 2. TripletMarginLoss ----------------------------------------------
    print("\n=== TripletMarginLoss ===")
    triplet = TripletMarginLoss(margin=0.2)
    loss_val = triplet(embeddings, labels)
    print(f"  Loss (margin=0.2): {float(loss_val):.6f}")

    for margin in [0.1, 0.2, 0.5, 1.0]:
        tl = TripletMarginLoss(margin=margin)
        val = tl(embeddings, labels)
        print(f"  margin={margin:.1f}  loss={float(val):.6f}")

    # -- 3. NTXentLoss (InfoNCE) -------------------------------------------
    print("\n=== NTXentLoss (InfoNCE) ===")
    ntxent = NTXentLoss(temperature=0.5)
    loss_val = ntxent(embeddings, labels)
    print(f"  Loss (temperature=0.5): {float(loss_val):.6f}")

    for temp in [0.1, 0.5, 1.0, 2.0]:
        nl = NTXentLoss(temperature=temp)
        val = nl(embeddings, labels)
        print(f"  temperature={temp:.1f}  loss={float(val):.6f}")
    print("  Lower temperature sharpens the softmax distribution.")

    # -- 4. ArcFaceLoss (learned weight matrix) ----------------------------
    print("\n=== ArcFaceLoss ===")
    num_classes = 4
    arcface = ArcFaceLoss(
        num_classes=num_classes,
        embedding_dim=embedding_dim,
        margin=0.5,
        scale=64.0,
        rngs=nnx.Rngs(0),
    )
    loss_val = arcface(embeddings, labels)
    print(f"  Loss (margin=0.5, scale=64): {float(loss_val):.6f}")

    # Show that ArcFace has trainable parameters
    graph_def, state = nnx.split(arcface)
    param_count = sum(p.size for p in jax.tree.leaves(state))
    print(f"  Trainable parameters: {param_count}")
    print(f"  Weight matrix shape: {arcface._weight[...].shape}")

    # -- 5. Gradient verification ------------------------------------------
    print("\n=== Gradient Verification ===")

    # Contrastive: verify grad flows through pure-function losses
    def contrastive_loss_fn(emb: jax.Array) -> jax.Array:
        return ContrastiveLoss(margin=1.0)(emb, labels)

    grad_fn = jax.grad(contrastive_loss_fn)
    grads = grad_fn(embeddings)
    grad_norm = float(jnp.linalg.norm(grads))
    print(f"  ContrastiveLoss gradient norm: {grad_norm:.6f}")
    print(f"  Gradient shape: {grads.shape}")
    has_nonzero = float(jnp.sum(jnp.abs(grads) > 1e-10))
    print(f"  Non-zero gradient elements: {int(has_nonzero)}/{grads.size}")

    # Triplet: verify grad flows
    def triplet_loss_fn(emb: jax.Array) -> jax.Array:
        return TripletMarginLoss(margin=0.2)(emb, labels)

    triplet_grads = jax.grad(triplet_loss_fn)(embeddings)
    print(f"\n  TripletMarginLoss gradient norm: {float(jnp.linalg.norm(triplet_grads)):.6f}")

    # NTXent: verify grad flows
    def ntxent_loss_fn(emb: jax.Array) -> jax.Array:
        return NTXentLoss(temperature=0.5)(emb, labels)

    ntxent_grads = jax.grad(ntxent_loss_fn)(embeddings)
    print(f"  NTXentLoss gradient norm: {float(jnp.linalg.norm(ntxent_grads)):.6f}")

    # ArcFace: verify grad flows through nnx.Module (capture model in closure)
    def arcface_loss_fn(emb: jax.Array) -> jax.Array:
        return arcface(emb, labels)

    arcface_grads = jax.grad(arcface_loss_fn)(embeddings)
    print(f"  ArcFaceLoss gradient norm: {float(jnp.linalg.norm(arcface_grads)):.6f}")

    # -- 6. Hard Negative Mining -------------------------------------------
    print("\n=== HardNegativeMiner ===")
    miner = HardNegativeMiner()
    mined = miner.mine(embeddings, labels)

    print(f"  Mined triplets: {len(mined.anchors)}")
    if len(mined.anchors) > 0:
        print("  First 5 triplets (anchor, positive, negative):")
        for i in range(min(5, len(mined.anchors))):
            a_idx = int(mined.anchors[i])
            p_idx = int(mined.positives[i])
            n_idx = int(mined.negatives[i])
            print(
                f"    ({a_idx}, {p_idx}, {n_idx}) -- "
                f"labels=({int(labels[a_idx])}, {int(labels[p_idx])}, {int(labels[n_idx])})"
            )
        # Verify mined triplets are valid
        all_valid = all(
            labels[int(mined.anchors[i])] == labels[int(mined.positives[i])]
            and labels[int(mined.anchors[i])] != labels[int(mined.negatives[i])]
            for i in range(len(mined.anchors))
        )
        print(f"  All triplets valid: {all_valid}")

    # Show that mined negatives are the hardest (closest different class)
    if len(mined.anchors) > 0:
        sample_anchor = int(mined.anchors[0])
        sample_neg = int(mined.negatives[0])
        anchor_emb = embeddings[sample_anchor]
        neg_emb = embeddings[sample_neg]
        dist = float(jnp.linalg.norm(anchor_emb - neg_emb))
        print("\n  Example hard negative:")
        print(f"    Anchor idx={sample_anchor} (label={int(labels[sample_anchor])})")
        print(f"    Hard neg idx={sample_neg} (label={int(labels[sample_neg])})")
        print(f"    Distance: {dist:.4f}")


if __name__ == "__main__":
    main()